In [ ]:
!pip install kagglehub
import kagglehub

# Download latest version
path = kagglehub.dataset_download("darrylljk/spotify-tracks")

print("Path to dataset files:", path)

In [ ]:
import os
import pandas as pd

files = os.listdir(path)
print('file yang tersedia:', files)

csv_file = os.path.join(path, files[0])
df= pd.read_csv(csv_file)

#cek sblm pembersihan
print('missing values')
print(df.isnull().sum())

df.drop_duplicates(inplace=True) # dikasi inplace biar dia ikut keubah
df.dropna(inplace=True)
print(df.isnull().sum())

print('data info')
df.info()

print('statistik deskripsi')
display(df.describe()) # display tu buat nunjukin, mirip print tp interaktif

print('5 baris pertama')
display(df.head())


In [ ]:
kolom = ['popularity', 'danceability', 'mode', 'tempo', 'energy']
korelasi = df[kolom].corr().round(2)
display(korelasi)

berati ini bacanya, korelasi antara popularity dengan danceability itu 0.04 (lemah), abistu popularity thdp mode -0.01(gada korelasi), popularity thdp tempo 0.01 (lemah), dan popularity thdp energy 0.00 (gada korelasi). korelasi sedang itu antara 0.04-0.05, trs 0.06-1.00 itu kuat.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8,6))
sns.heatmap(korelasi, cmap='coolwarm', annot=True, fmt='.2f') # korelasi itu nama variabel yg di corr(), abistu cmap itu buat nentuin warna nya, annot itu buat munculin angka di dlm kotak, fmt='.2f' itu buat bikin 2 angka di blkg koma
plt.title('korelasi antara popularity, danceability, tempo, mode, dan energy', fontsize= 11, color='black')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8,6))
sns.scatterplot(data = df, x= 'popularity', y='tempo')
plt.title('korelasi antara popularity dan tempo', fontsize=11, color='black')
plt.grid(color='darkgray', linestyle='-', linewidth=0.5)
plt.tight_layout()
plt.show()


In [ ]:
hasil_genre= df.groupby('track_genre')['popularity'].mean() #group by berdasarkan track dari kolom popularity trs cari rata" nya
top_genre= hasil_genre.sort_values(ascending=False).round(2).head()
display(top_genre)


In [ ]:
plt.figure(figsize=(8,6))
top_genre.plot(kind='bar', color='skyblue') #lgsg aja dipanggil pake plot, karna kl pake data= itu bingung karna top_genre udh berupa index
plt.title('top 5 genre lagu populer', fontsize=11)
plt.xlabel('most populer')
plt.ylabel('rata" popularitas')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error # metrics regresi

x = df[['danceability', 'tempo', 'energy']]
y= df['popularity']

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.30, random_state=42)

model = LinearRegression()
model.fit(x_train, y_train)
y_pred_lr = model.predict(x_test)

a = RandomForestRegressor(random_state=42)
a.fit(x_train, y_train)
y_pred_rf = a.predict(x_test)

print('r2 skor: ', r2_score(y_test, y_pred_lr))
print('mse eror: ', mean_squared_error(y_test, y_pred_lr)) # untuk ngelihat rata" selisih tebakan
print('r2 skor: ', r2_score(y_test, y_pred_rf))
print('mse eror: ', mean_squared_error(y_test, y_pred_rf)) 

LAPORAN ANALISIS KORELASI ANTARA POPULARITY TERHADAP DANCEABILITY, TEMPO, MODE, dan ENERGY
berdasarkan heatmap plot dapat disimpulkan bahwa korelasi antara popularity terhadap danceability, tempo, mode, dan energy tidak memiliki hubungan yang kuat, atau bisa dikatakan ngga ada korelasi nya sama sekali.
kemudian untuk data scatter plot antara popularity dengan tempo juga tidak ada hubungan yang erat, tersebar secara acak. 
untuk track genre top 5 diduduki oleh pop-fil, kpop, chill, sad dan grunge.
lalu untuk algoritma linear regresiion didapatkan angka 0.0015 dengan metrik r2 dan mse eror nya 495.155
untuk algoritma randomforestregressor angka 0.416 di r2 dan 289.495 di selisih tebakan.
dengan ini disimpulkan bahwa menggunakan algoritma randomforestregressor lebih disarankan, karena menguranggi eror dan meningkatkan angka r2 nya.